# 03 · Limpieza y transformación (ETL)
**Fase CRISP-DM:** 3. Preparación de los datos

**ETL** significa:
- **E**xtract (Extraer) → leer los datos originales de `data/raw`.
- **T**ransform (Transformar) → limpiar errores y crear nuevas columnas útiles.
- **L**oad (Cargar) → guardar el resultado en `data/processed` (y después en SQL Server).

> El archivo original **no se modifica**: todo el resultado se guarda en un archivo nuevo.

## 1. Extract: leer los datos originales

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA_RAW = Path("..") / "data" / "raw" / "abalone.data"
RUTA_PROC = Path("..") / "data" / "processed"
RUTA_PROC.mkdir(parents=True, exist_ok=True)

COLUMNAS = ["Sex", "Length", "Diameter", "Height", "Whole_weight",
            "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"]
df_raw = pd.read_csv(RUTA_RAW, header=None, names=COLUMNAS)
df = df_raw.copy()   # trabajamos sobre una COPIA, el original queda intacto
print("Registros originales:", len(df))

Registros originales: 4177


## 2. Transform – Paso A: eliminar registros físicamente imposibles
Reglas definidas a partir del EDA (notebook 02). Guardamos **cuántos registros elimina cada regla** para documentarlo.

In [2]:
reglas = {
    "Altura igual a 0":                       df["Height"] == 0,
    "Altura extrema (> 0.3)":                 df["Height"] > 0.3,
    "Peso de carne mayor que el peso total":  df["Shucked_weight"] > df["Whole_weight"],
    "Peso de concha mayor que el peso total": df["Shell_weight"] > df["Whole_weight"],
}

bitacora = []
mascara_eliminar = pd.Series(False, index=df.index)
for nombre, condicion in reglas.items():
    bitacora.append({"Regla": nombre, "Registros que cumplen": int(condicion.sum())})
    mascara_eliminar = mascara_eliminar | condicion

print(pd.DataFrame(bitacora))
print("\nTotal de registros únicos a eliminar:", int(mascara_eliminar.sum()))

df = df[~mascara_eliminar].reset_index(drop=True)
print("Registros después de limpiar:", len(df))

                                    Regla  Registros que cumplen
0                        Altura igual a 0                      2
1                  Altura extrema (> 0.3)                      2
2   Peso de carne mayor que el peso total                      4
3  Peso de concha mayor que el peso total                      1

Total de registros únicos a eliminar: 8
Registros después de limpiar: 4169


## 3. Transform – Paso B: nombres descriptivos para Sex
Para el dashboard es más claro leer "Macho" que "M".

In [3]:
df["Sexo"] = df["Sex"].map({"M": "Macho", "F": "Hembra", "I": "Infante"})
df[["Sex", "Sexo"]].drop_duplicates()

,Sex,Sexo
0,M,Macho
2,F,Hembra
4,I,Infante


## 4. Transform – Paso C: convertir a unidades reales
Multiplicamos por 200 para tener **mm** y **gramos** reales. Se usarán en el Data Mart y en Power BI; las columnas escaladas originales se conservan para los modelos.

In [4]:
MEDIDAS = ["Length", "Diameter", "Height"]
PESOS = ["Whole_weight", "Shucked_weight", "Viscera_weight", "Shell_weight"]

for col in MEDIDAS:
    df[col + "_mm"] = (df[col] * 200).round(1)
for col in PESOS:
    df[col + "_g"] = (df[col] * 200).round(1)

df[[c + "_mm" for c in MEDIDAS] + [c + "_g" for c in PESOS]].describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
Length_mm,4169.0,104.8,24.0,15.0,90.0,109.0,123.0,163.0
Diameter_mm,4169.0,81.6,19.8,11.0,70.0,85.0,96.0,130.0
Height_mm,4169.0,27.9,7.7,2.0,23.0,28.0,33.0,50.0
Whole_weight_g,4169.0,165.9,98.0,0.4,88.5,160.1,230.8,565.1
Shucked_weight_g,4169.0,71.9,44.4,0.2,37.2,67.2,100.4,297.6
Viscera_weight_g,4169.0,36.1,21.9,0.1,18.7,34.2,50.6,152.0
Shell_weight_g,4169.0,47.8,27.8,0.3,26.0,46.8,65.8,201.0


## 5. Transform – Paso D: crear la edad y el grupo de edad
- **Edad_anios** = Rings + 1.5 (según la documentación del dataset).
- **Grupo_Edad** = variable para la **clasificación**. Criterio de corte documentado en la Unidad I, basado en los cuartiles de Rings:

| Grupo | Criterio |
|---|---|
| Joven | Rings ≤ 8 |
| Adulto | 9 ≤ Rings ≤ 10 |
| Maduro | Rings ≥ 11 |

In [5]:
df["Edad_anios"] = df["Rings"] + 1.5

df["Grupo_Edad"] = pd.cut(
    df["Rings"],
    bins=[0, 8, 10, np.inf],          # (0, 8]  (8, 10]  (10, ∞)
    labels=["Joven", "Adulto", "Maduro"],
).astype(str)

tabla = df["Grupo_Edad"].value_counts().reindex(["Joven", "Adulto", "Maduro"])
print(pd.DataFrame({"Registros": tabla, "Porcentaje": (tabla / len(df) * 100).round(1)}))

            Registros  Porcentaje
Grupo_Edad                       
Joven            1400        33.6
Adulto           1322        31.7
Maduro           1447        34.7


Las tres clases quedan **balanceadas** (≈ 33% cada una), lo que evita que un modelo de clasificación favorezca a una sola clase.

## 6. Transform – Paso E: identificador único
Cada abulón necesita un **ID** para el Data Mart (llave primaria).

In [6]:
df.insert(0, "ID_Abulon", range(1, len(df) + 1))
df.head()

,ID_Abulon,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings,Sexo,Length_mm,Diameter_mm,Height_mm,Whole_weight_g,Shucked_weight_g,Viscera_weight_g,Shell_weight_g,Edad_anios,Grupo_Edad
0,1,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,Macho,91.0,73.0,19.0,102.8,44.9,20.2,30.0,16.5,Maduro
1,2,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,Macho,70.0,53.0,18.0,45.1,19.9,9.7,14.0,8.5,Joven
2,3,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,Hembra,106.0,84.0,27.0,135.4,51.3,28.3,42.0,10.5,Adulto
3,4,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,Macho,88.0,73.0,25.0,103.2,43.1,22.8,31.0,11.5,Adulto
4,5,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,Infante,66.0,51.0,16.0,41.0,17.9,7.9,11.0,8.5,Joven


## 7. Validaciones finales
Comprobamos que el resultado sea correcto antes de guardarlo. `assert` detiene el notebook con un error si algo no se cumple.

In [7]:
assert df.isna().sum().sum() == 0, "Hay valores nulos"
assert (df["Height"] > 0).all(), "Hay alturas inválidas"
assert (df["Shucked_weight"] <= df["Whole_weight"]).all()
assert (df["Shell_weight"] <= df["Whole_weight"]).all()
assert df["ID_Abulon"].is_unique
assert len(df_raw) == 4177, "¡El original cambió!"
print("✅ Todas las validaciones pasaron")
print("Registros finales:", len(df), "| Columnas:", df.shape[1])

✅ Todas las validaciones pasaron
Registros finales: 4169 | Columnas: 20


## 8. Load: guardar en data/processed

In [8]:
ruta_salida = RUTA_PROC / "abalone_limpio.csv"
# lineterminator="\n" para que SQL Server (BULK INSERT) lea bien el archivo también en Windows
df.to_csv(ruta_salida, index=False, encoding="utf-8", lineterminator="\n")
print("Archivo guardado en:", ruta_salida)
print(list(df.columns))

Archivo guardado en: ../data/processed/abalone_limpio.csv
['ID_Abulon', 'Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings', 'Sexo', 'Length_mm', 'Diameter_mm', 'Height_mm', 'Whole_weight_g', 'Shucked_weight_g', 'Viscera_weight_g', 'Shell_weight_g', 'Edad_anios', 'Grupo_Edad']


## 9. Resumen del ETL
| Etapa | Resultado |
|---|---|
| Extract | 4,177 registros leídos de `data/raw/abalone.data` |
| Transform | Eliminación de registros físicamente imposibles, sexo descriptivo, unidades reales, Edad_anios, Grupo_Edad e ID |
| Load | `data/processed/abalone_limpio.csv` (siguiente paso: cargarlo en el Data Mart de SQL Server) |